In [ ]:
!pip install oracledb

In [ ]:
import oracledb
import pandas as pd

In [ ]:
# connect to Oracle
conn = oracledb.connect(
    user="ora_CWL",
    password="aSTU_ID",
    dsn=oracledb.makedsn("localhost", 1522, service_name="stu")
)
cur = conn.cursor()

# optional: drop old tables first if they already exist
for table_name in [
    "Letterboxd_Genres",
    "Letterboxd_Studios",
    "Letterboxd_Movies",
    "IMDB_Movies",
    "Rotten_Tomatoes_Movies"
]:
    try:
        cur.execute(f"DROP TABLE {table_name} CASCADE CONSTRAINTS")
    except oracledb.DatabaseError:
        pass

# LETTERBOXD MOVIES
cur.execute("""
CREATE TABLE Letterboxd_Movies (
    id NUMBER PRIMARY KEY,
    name VARCHAR2(200) NOT NULL,
    scaled_rating NUMBER NOT NULL,
    release_year NUMBER NOT NULL
)
""")

# LETTERBOXD GENRES
cur.execute("""
CREATE TABLE Letterboxd_Genres (
    id NUMBER,
    genre_score NUMBER NOT NULL,
    genre VARCHAR2(200) NOT NULL,
    PRIMARY KEY (id, genre),
    FOREIGN KEY (id)
        REFERENCES Letterboxd_Movies(id)
        ON DELETE CASCADE
)
""")

# LETTERBOXD STUDIOS
cur.execute("""
CREATE TABLE Letterboxd_Studios (
    id NUMBER,
    studio VARCHAR2(200) NOT NULL,
    PRIMARY KEY (id, studio),
    FOREIGN KEY (id)
        REFERENCES Letterboxd_Movies(id)
        ON DELETE CASCADE
)
""")

# IMDB MOVIES
cur.execute("""
CREATE TABLE IMDB_Movies (
    genre_score NUMBER NOT NULL,
    name VARCHAR2(200) NOT NULL,
    scaled_score NUMBER NOT NULL,
    genre VARCHAR2(100),
    year NUMBER NOT NULL,
    company VARCHAR2(200) NOT NULL,
    PRIMARY KEY (name, year)
)
""")

# ROTTEN TOMATOES MOVIES
cur.execute("""
CREATE TABLE Rotten_Tomatoes_Movies (
    movie_title VARCHAR2(200) NOT NULL,
    year NUMBER NOT NULL,
    genre_score NUMBER NOT NULL,
    genre_split VARCHAR2(200),
    scaled_tomatometer_rating NUMBER NOT NULL,
    studio_name VARCHAR2(200) NOT NULL,
    PRIMARY KEY (movie_title, year)
)
""")

conn.commit()
print("Workflow 3 complete: tables created.")

In [ ]:
cur.execute("DELETE FROM Letterboxd_Genres")
cur.execute("DELETE FROM Letterboxd_Studios")
cur.execute("DELETE FROM Letterboxd_Movies")
cur.execute("DELETE FROM IMDB_Movies")
cur.execute("DELETE FROM Rotten_Tomatoes_Movies")

conn.commit()

In [ ]:
# read cleaned CSV files
df_letterboxd_movies = pd.read_csv("selected_letterboxd_movies.csv")
df_letterboxd_genres = pd.read_csv("selected_letterboxd_genres.csv")
df_letterboxd_studios = pd.read_csv("studios.csv")
df_imdb_movies = pd.read_csv("selected_movie_industry.csv")
df_rotten = pd.read_csv("selected_rotten_tomato_movies.csv")

# drop NA rows
df_letterboxd_movies = df_letterboxd_movies.dropna()
df_letterboxd_genres = df_letterboxd_genres.dropna()
df_letterboxd_studios = df_letterboxd_studios.dropna()
df_imdb_movies = df_imdb_movies.dropna()
df_rotten = df_rotten.dropna()

# remove duplicates that violate primary keys
df_letterboxd_studios = df_letterboxd_studios.drop_duplicates(subset=["id","studio"])
df_letterboxd_genres = df_letterboxd_genres.drop_duplicates(subset=["id","genre"])
df_letterboxd_movies = df_letterboxd_movies.drop_duplicates(subset=["id"])
df_imdb_movies = df_imdb_movies.drop_duplicates(subset=["name","year"])
df_rotten = df_rotten.drop_duplicates(subset=["movie_title","year"])

# limit dataset size to avoid Oracle quota error
df_letterboxd_movies = df_letterboxd_movies.head(3000)
df_letterboxd_genres = df_letterboxd_genres.head(3000)
df_letterboxd_studios = df_letterboxd_studios.head(3000)
df_imdb_movies = df_imdb_movies.head(3000)
df_rotten = df_rotten.head(3000)

# keep only genres/studios whose movie id exists
valid_movie_ids = set(df_letterboxd_movies["id"])

df_letterboxd_genres = df_letterboxd_genres[
    df_letterboxd_genres["id"].isin(valid_movie_ids)
]

df_letterboxd_studios = df_letterboxd_studios[
    df_letterboxd_studios["id"].isin(valid_movie_ids)
]

# LETTERBOXD MOVIES
movies_data = [
    (
        int(r["id"]),
        str(r["name"]),
        float(r["scaled_rating"]),
        int(r["date"])
    )
    for _, r in df_letterboxd_movies.iterrows()
]

cur.executemany("""
INSERT INTO Letterboxd_Movies (id, name, scaled_rating, release_year)
VALUES (:1, :2, :3, :4)
""", movies_data)

print("Inserted Letterboxd_Movies")


# LETTERBOXD GENRES
genres_data = [
    (
        int(r["id"]),
        float(r["genre_score"]),
        str(r["genre"])
    )
    for _, r in df_letterboxd_genres.iterrows()
]

cur.executemany("""
INSERT INTO Letterboxd_Genres (id, genre_score, genre)
VALUES (:1, :2, :3)
""", genres_data)

print("Inserted Letterboxd_Genres")


# LETTERBOXD STUDIOS
studios_data = [
    (
        int(r["id"]),
        str(r["studio"])
    )
    for _, r in df_letterboxd_studios.iterrows()
]

cur.executemany("""
INSERT INTO Letterboxd_Studios (id, studio)
VALUES (:1, :2)
""", studios_data)

print("Inserted Letterboxd_Studios")


# IMDB MOVIES
imdb_data = [
    (
        float(r["genre_score"]),
        str(r["name"]),
        float(r["scaled_score"]),
        str(r["genre"]),
        int(r["year"]),
        str(r["company"])
    )
    for _, r in df_imdb_movies.iterrows()
]

cur.executemany("""
INSERT INTO IMDB_Movies (genre_score, name, scaled_score, genre, year, company)
VALUES (:1, :2, :3, :4, :5, :6)
""", imdb_data)

print("Inserted IMDB_Movies")


# ROTTEN TOMATOES
rotten_data = [
    (
        str(r["movie_title"]),
        int(r["year"]),
        float(r["genre_score"]),
        str(r["genre_split"]),
        float(r["scaled_tomatometer_rating"]),
        str(r["studio_name"])
    )
    for _, r in df_rotten.iterrows()
]

cur.executemany("""
INSERT INTO Rotten_Tomatoes_Movies
(movie_title, year, genre_score, genre_split, scaled_tomatometer_rating, studio_name)
VALUES (:1, :2, :3, :4, :5, :6)
""", rotten_data)

print("Inserted Rotten_Tomatoes_Movies")


# commit everything
conn.commit()

print("Workflow 4 complete: all data inserted successfully.")

In [ ]:
def run_query(sql):
    cur.execute(sql)
    cols = [c[0] for c in cur.description]
    rows = cur.fetchall()
    return pd.DataFrame(rows, columns=cols)

In [ ]:
run_query("SELECT COUNT(*) FROM Letterboxd_Movies")

In [ ]:
run_query("SELECT COUNT(*) FROM Letterboxd_Genres")

In [ ]:
run_query("SELECT COUNT(*) FROM Letterboxd_Studios")

In [ ]:
run_query("SELECT COUNT(*) FROM IMDB_Movies")

In [ ]:
run_query("SELECT COUNT(*) FROM Rotten_Tomatoes_Movies")

In [ ]:
df_a = run_query("""
SELECT
    TRUNC(mi.year/5)*5 AS year_interval,
    AVG(mi.scaled_score) AS avg_imdb_rating,
    AVG(lm.scaled_rating) AS avg_letterboxd_rating,
    AVG(rt.scaled_tomatometer_rating) AS avg_rotten_rating
FROM IMDB_Movies mi
JOIN Rotten_Tomatoes_Movies rt
    ON mi.name = rt.movie_title
   AND mi.year = rt.year
JOIN Letterboxd_Movies lm
    ON mi.name = lm.name
   AND mi.year = lm.release_year
WHERE mi.year BETWEEN 1990 AND 2015
GROUP BY TRUNC(mi.year/5)*5
ORDER BY year_interval
""")

df_a

In [ ]:

# df_a = run_query("""
# SELECT
#     mi.year,
#     AVG(mi.scaled_score) AS avg_imdb_rating,
#     AVG(lm.scaled_rating) AS avg_letterboxd_rating,
#     AVG(rt.scaled_tomatometer_rating) AS avg_rotten_rating
# FROM IMDB_Movies mi
# JOIN Rotten_Tomatoes_Movies rt
#     ON mi.name = rt.movie_title
#    AND mi.year = rt.year
# JOIN Letterboxd_Movies lm
#     ON mi.name = lm.name
#    AND mi.year = lm.release_year
# WHERE mi.year BETWEEN 1990 AND 2015
# GROUP BY mi.year
# ORDER BY mi.year
# """)

# df_a

In [ ]:
df_b = run_query("""
SELECT genre_category,
		   AVG(lm.scaled_rating) AS avg_audience_rating,
		   COUNT(*) AS total_films
FROM (SELECT lg.id,
			       AVG(lg.genre_score) AS avg_genre_score,
			       'emotionally_positive' AS genre_category
	    FROM selected_letterboxd_genres lg
	    GROUP BY lg.id
	    HAVING AVG(lg.genre_score) > 0

	    UNION ALL

	    SELECT lg.id,
			       AVG(lg.genre_score) AS avg_genre_score,
			       'fear_based' AS genre_category
		  FROM selected_letterboxd_genres lg
	    GROUP BY lg.id
	    HAVING AVG(lg.genre_score) < 0
			) AS categorized_films
JOIN selected_letterboxd_movies lm
    ON categorized_films.id = lm.id
GROUP BY genre_category;
""")

df_b

In [ ]:
df_c = run_query("""
SELECT 'major' AS studio_group,
       AVG(lm.scaled_rating) AS avg_rating,
       STDDEV(lm.scaled_rating) AS rating_variability,
       COUNT(DISTINCT lm.id) AS total_films
FROM Letterboxd_Movies lm
JOIN Letterboxd_Studios ls
    ON lm.id = ls.id
WHERE ls.studio IN (
    SELECT studio
    FROM Letterboxd_Studios
    GROUP BY studio
    HAVING COUNT(DISTINCT id) >= 20
)

UNION

SELECT 'small' AS studio_group,
       AVG(lm.scaled_rating) AS avg_rating,
       STDDEV(lm.scaled_rating) AS rating_variability,
       COUNT(DISTINCT lm.id) AS total_films
FROM Letterboxd_Movies lm
JOIN Letterboxd_Studios ls
    ON lm.id = ls.id
WHERE ls.studio IN (
    SELECT studio
    FROM Letterboxd_Studios
    GROUP BY studio
    HAVING COUNT(DISTINCT id) < 20
)
""")

df_c